In [1]:
# Clear pip cache and force reinstall dependencies


import hashlib
from Crypto.Hash import RIPEMD160
from ecdsa import SigningKey, SECP256k1
import cupy as cp
import os
import pickle
import secrets
import bitcoinaddress
from bitstring import BitArray
from google.colab import drive

# Try importing pybloomfiltermmap3, fallback if needed
try:
    from pybloomfiltermmap3 import BloomFilter
except ImportError:
    print("pybloomfiltermmap3 import failed. Please restart runtime and try again.")
    raise

# Mount Google Drive
drive.mount('/content/drive')
addresses_file = '/content/drive/MyDrive/addresses.txt'
bloom_hash160_file = '/content/drive/MyDrive/bloom_hash160.bf'
bloom_keys_file = '/content/drive/MyDrive/bloom_keys.bf'
keys_file = '/content/drive/MyDrive/generated_keys.txt'
matches_file = '/content/drive/MyDrive/matches.txt'
checkpoint_file = '/content/drive/MyDrive/checkpoint.pkl'

# Secp256k1 curve order (max private key value)
curve_order = 0xFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFEBAAEDCE6AF48A03BBFD25E8CD0364141

# Read and decode 999 P2PKH addresses to Hash160s
def read_addresses(file_path):
    hash160s = []
    with open(file_path, 'r') as f:
        for line in f:
            addr = line.strip()
            try:
                wallet = bitcoinaddress.Wallet(addr)
                hash160 = wallet.hash160
                hash160s.append(hash160)
            except Exception as e:
                print(f"Invalid address {addr}: {e}")
    return hash160s

# Hamming distance function
def hamming_distance(a, b):
    return (BitArray(a) ^ BitArray(b)).count('1')

# Load target Hash160s
try:
    target_hash160s = read_addresses(addresses_file)
    print(f"Loaded {len(target_hash160s)} Hash160s")
except FileNotFoundError:
    print(f"Error: {addresses_file} not found. Please upload to Google Drive.")
    raise

# Bloom filters: Hash160s and keys (no duplicates)
keyspace_size = 2**30  # ~1.07 billion keys
try:
    bloom_hash160 = BloomFilter(keyspace_size, 0.01, bloom_hash160_file)  # ~1.58 GB
    bloom_keys = BloomFilter(keyspace_size, 0.01, bloom_keys_file)  # ~1.58 GB
except Exception as e:
    print(f"Bloom filter initialization failed: {e}")
    raise

# GPU-accelerated P2PKH computation
def gpu_p2pkh(keys):
    try:
        keys_gpu = cp.array([int(k, 16) for k in keys])
        pubkeys = [SigningKey.from_string(k.to_bytes(32, "big"), curve=SECP256k1).verifying_key.to_string() for k in keys]
        pubkeys_gpu = cp.array(pubkeys)
        hash160s = cp.array([RIPEMD160.new(pk).digest() for pk in pubkeys_gpu.get()])
        matches = []
        for h in hash160s.get():
            bloom_hash160.add(h)
            for target in target_hash160s:
                if h in bloom_hash160 and hamming_distance(h, target) < 30:
                    matches.append((h, target))
        return matches
    except Exception as e:
        print(f"GPU computation failed: {e}")
        return []

# Generate non-sequential, non-duplicate keys
def generate_seed_key():
    while True:
        key = secrets.randbelow(curve_order) + 1  # 1 to curve_order
        key_hex = hex(key)[2:].zfill(64)
        if key_hex not in bloom_keys:
            bloom_keys.add(key_hex)
            return key_hex

# Save keys to file
def save_keys(keys, indices):
    with open(keys_file, 'a') as f:
        for idx, key in zip(indices, keys):
            f.write(f"{idx}:{key}\n")

# Main loop
batch_size = 10**6  # ~1M keys per batch, fits 40 GB HBM3
keys_processed = 0

# Load checkpoint
if os.path.exists(checkpoint_file):
    with open(checkpoint_file, 'rb') as f:
        keys_processed = pickle.load(f)

batch_index = 0
while keys_processed < keyspace_size:
    batch_keys = []
    batch_indices = []
    for _ in range(batch_size):
        if keys_processed >= keyspace_size:
            break
        key = generate_seed_key()
        batch_keys.append(key)
        batch_indices.append(keys_processed)
        keys_processed += 1

    # Save keys
    try:
        save_keys(batch_keys, batch_indices)
    except Exception as e:
        print(f"Failed to save keys: {e}")

    # Compute Hash160s
    matches = gpu_p2pkh(batch_keys)

    # Save matches
    if matches:
        try:
            with open(matches_file, 'a') as f:
                for h, target in matches:
                    f.write(f"Match: Hash160={h.hex()}, Target={target.hex()}, Index={batch_indices[0]}\n")
            print(f"Matches found: {len(matches)}")
        except Exception as e:
            print(f"Failed to save matches: {e}")

    # Save checkpoint
    try:
        with open(checkpoint_file, 'wb') as f:
            pickle.dump(keys_processed, f)
    except Exception as e:
        print(f"Failed to save checkpoint: {e}")

    batch_index += 1
    print(f"Processed {keys_processed}/{keyspace_size} keys ({100 * keys_processed / keyspace_size:.2f}%)")

print("Done")

pybloomfiltermmap3 import failed. Please restart runtime and try again.


ModuleNotFoundError: No module named 'pybloomfiltermmap3'